# Build the analysis inputs from the GEO files

Put the files downloaded from GEO in `geo/` and run this notebook from the repository root. It writes the processed zarrs (the preprocessed one is the annotated one without the results of notebook 01), the 00 results read by the notebooks, and `xenium_data_<species>/integrated/integrated_<species>.zarr`: the count table of `integrated_<species>.h5ad` with the images, outlines and transcripts of the sample files, placed back on the slide (mouse slide No2 to the right of No1, as in `merge-rois-mouse.ipynb`).

In [ ]:
import os
import re
import shutil

import anndata as ad
import dask.array as da
import dask.dataframe as dd
import geopandas as gpd
import numpy as np
import pandas as pd
import shapely
import spatialdata
import tifffile
import zarr
from spatialdata._io.format import SpatialDataContainerFormatV01
from spatialdata.models import Image2DModel, PointsModel, ShapesModel, TableModel
from spatialdata.transformations import Identity, Scale, Sequence, Translation, set_transformation

ad.settings.zarr_write_format = 2

GEO = "geo"
PIXEL_SIZE = 0.2125
SLIDE_SHIFT = {"No1": 0.0, "No2": 9865.9248046875}
# top-left pixel of each sample on its slide
OFFSET_PX = {
    "human": {
        "C-ion_Late": (18227, 86802),
        "Xray_Late": (5376, 76842),
        "SF1": (4314, 92591),
        "SF2": (31751, 46453),
        "Xray_Under": (19573, 71380),
    },
    "mouse": {
        "C-ion_Sham_No1": (15457, 88899),
        "C-ion_24_h_No1": (3171, 94969),
        "C-ion_48_h_No1": (6345, 80776),
        "C-ion_3.5_d_No1": (4828, 61309),
        "gamma-ray_Sham_No1": (15205, 69701),
        "gamma-ray_24_h_No1": (26599, 95764),
        "gamma-ray_48_h_No1": (28886, 79166),
        "gamma-ray_3.5_d_No1": (30838, 60971),
        "C-ion_24_h_No2": (6702, 98178),
        "C-ion_48_h_No2": (2980, 80258),
        "C-ion_3.5_d_No2": (4216, 65870),
        "gamma-ray_24_h_No2": (31750, 92793),
        "gamma-ray_48_h_No2": (35114, 76644),
        "gamma-ray_3.5_d_No2": (18643, 71221),
    },
}

In [ ]:
# added to the preprocessed object by 01_export_and_annotate.ipynb
ANNOTATION = {
    "human": ["cell_subtype", "cell_largetype"],
    "mouse": ["leiden_mixed", "leiden_sub", "leiden", "cell_subtype", "cell_largetype"],
}

for s in ("human", "mouse"):
    os.makedirs(f"{s}_analysis/data", exist_ok=True)
    adata = ad.read_h5ad(f"{GEO}/integrated_{s}_annotated.h5ad")
    adata.write_zarr(f"{s}_analysis/data/integrated_{s}_annotated.zarr")
    adata.obs = adata.obs.drop(columns=ANNOTATION[s])
    for k in ("leiden_mixed", "leiden_sub", "rank_genes_groups"):
        adata.uns.pop(k, None)
    adata.write_zarr(f"{s}_analysis/data/integrated_{s}_preprocessed.zarr")
    res = f"{s}_analysis/results/00_preprocess_and_cluster"
    os.makedirs(res, exist_ok=True)
    shutil.copy(f"{GEO}/{s}_qc_per_cell_before.parquet", f"{res}/qc_per_cell_before.parquet")
shutil.copy(f"{GEO}/mouse_roi_numbers.csv", "mouse_analysis/results/00_preprocess_and_cluster/roi_numbers.csv")

In [ ]:
def outlines(path, keep, dx, dy, roi):
    v = pd.read_parquet(path)
    v = v[v["cell_id"].isin(keep)].sort_values("cell_id", kind="stable")
    # back to float32, the type of the XOA coordinates
    xy = np.c_[(v["vertex_x"] + dx).astype(np.float32), (v["vertex_y"] + dy).astype(np.float32)].astype(float)
    ids, start = np.unique(v["cell_id"].values, return_index=True)
    polygons = shapely.from_ragged_array(shapely.GeometryType.POLYGON, xy, (np.r_[start, len(v)], np.arange(len(ids) + 1)))
    return gpd.GeoDataFrame(geometry=polygons, index=pd.Index(ids).astype(str) + f"_{roi}")


def transcripts(path, left_out, dx, dy, roi):
    t = dd.read_parquet(path).rename(columns={"x_location": "x", "y_location": "y", "z_location": "z"})
    t = t[~t["cell_id"].isin(left_out)]
    return t.assign(x=(t["x"] + dx).astype(np.float32), y=(t["y"] + dy).astype(np.float32), roi_id=roi)


def image(path, dx, dy):
    with tifffile.TiffFile(path) as f:
        channels = re.findall(r'<Channel [^>]*Name="([^"]*)"', f.ome_metadata)
    data = da.from_zarr(zarr.open(tifffile.imread(path, aszarr=True, level=0), mode="r")).rechunk((1, 4096, 4096))
    to_slide = Sequence([Scale([PIXEL_SIZE, PIXEL_SIZE], axes=("x", "y")), Translation([dx, dy], axes=("x", "y"))])
    return Image2DModel.parse(data, dims=("c", "y", "x"), c_coords=channels, transformations={"global": to_slide})

In [ ]:
for s in ("human", "mouse"):
    table = ad.read_h5ad(f"{GEO}/integrated_{s}.h5ad")
    images, cells, nuclei, points = {}, [], [], []
    for roi, (x0, y0) in OFFSET_PX[s].items():
        prefix = f"{GEO}/{s}_{roi}_"
        dx, dy = x0 * PIXEL_SIZE + SLIDE_SHIFT.get(roi[-3:], 0.0), y0 * PIXEL_SIZE
        keep = table.obs.loc[table.obs["roi_id"] == roi, "cell_id"].astype(str)
        left_out = list(set(pd.read_parquet(prefix + "cells.parquet")["cell_id"]) - set(keep))
        cells.append(outlines(prefix + "cell_boundaries.parquet", keep, dx, dy, roi))
        nuclei.append(outlines(prefix + "nucleus_boundaries.parquet", keep, dx, dy, roi))
        points.append(transcripts(prefix + "transcripts.parquet", left_out, dx, dy, roi))
        images[f"morphology_focus_{roi}"] = image(prefix + "morphology_focus.ome.tif", dx, dy)

    cell_boundaries = ShapesModel.parse(pd.concat(cells).loc[table.obs_names])
    nucleus_boundaries = ShapesModel.parse(pd.concat(nuclei))
    points = PointsModel.parse(dd.concat(points), coordinates={"x": "x", "y": "y", "z": "z"},
                               feature_key="feature_name", instance_key="cell_id")
    for element in (cell_boundaries, nucleus_boundaries, points):
        set_transformation(element, {"global": Identity()}, set_all=True)

    sdata = spatialdata.SpatialData(
        images=images,
        shapes={"cell_boundaries": cell_boundaries, "nucleus_boundaries": nucleus_boundaries},
        points={"transcripts": points},
        tables={"table": TableModel.parse(table)},
    )
    os.makedirs(f"xenium_data_{s}/integrated", exist_ok=True)
    sdata.write(f"xenium_data_{s}/integrated/integrated_{s}.zarr", sdata_formats=[SpatialDataContainerFormatV01()])